# LeetCode 3398 — Smallest Substring With Identical Characters I bằng Python

- Link: https://leetcode.com/problems/smallest-substring-with-identical-characters-i/
- Độ khó: **Hard**
- Chủ đề: **Tìm kiếm nhị phân đáp án** + **quy hoạch động máy trạng thái** `(ký tự cuối, độ dài khối cuối)` để tính số lần lật ít nhất cho mỗi giới hạn `L`

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def minLength(self, s: str, numOps: int) -> int:
```

## 1. Mục tiêu học

1. Biến bài tối ưu thành **bài kiểm tra**: với `L` cho trước, số lần lật **ít nhất** để mọi khối giống nhau dài ≤ `L` là bao nhiêu?
2. Viết DP trên **máy trạng thái**: trạng thái `(c, r)` = ký tự cuối `c` và độ dài khối cuối `r ∈ [1, L]`; mỗi ký tự giữ nguyên (chi phí 0) hoặc lật (chi phí 1).
3. Dùng **tính đơn điệu** theo `L` để nhị phân, tổng `O(n · L · log n)` — đủ nhanh với `n ≤ 1000`.
4. Hiểu công thức nhanh hơn (dùng cho bản II) và dùng DP để **kiểm chứng** nó, đặc biệt trường hợp `L = 1`.
5. Thực hành dựng lại **xâu kết quả** từ DP (truy vết).

## 2. Đề bài & ví dụ

Cho xâu nhị phân `s` và `numOps`. Mỗi thao tác lật một ký tự (`0 ↔ 1`), tối đa `numOps` lần. Tối thiểu hóa độ dài **xâu con dài nhất gồm các ký tự giống nhau**.

**Ví dụ 1:** `"000001", 1` → `2`. **Ví dụ 2:** `"0000", 2` → `1`. **Ví dụ 3:** `"0101", 0` → `1`.

**Ràng buộc:** `1 ≤ n ≤ 1000`, `0 ≤ numOps ≤ n`.

In [1]:
examples = [("000001", 1, 2), ("0000", 2, 1), ("0101", 0, 1)]
for s, k, exp in examples:
    print(repr(s), k, '-> kỳ vọng', exp)

'000001' 1 -> kỳ vọng 2
'0000' 2 -> kỳ vọng 1
'0101' 0 -> kỳ vọng 1


## 3. Kiến thức nền

### 3.1. Khối dài nhất

In [2]:
def longest_run(t):
    best = cur = 1
    for i in range(1, len(t)):
        cur = cur + 1 if t[i] == t[i - 1] else 1
        best = max(best, cur)
    return best

print(longest_run("000001"), longest_run("001001"), longest_run("1010"))

5 2 1


### 3.2. DP máy trạng thái

Đọc xâu từ trái; sau mỗi ký tự, trạng thái là `(c, r)`. Ký tự tiếp theo chọn `d ∈ {0, 1}` (chi phí `[d ≠ s[i]]`):

- `d = c` ⇒ `(c, r + 1)`, chỉ hợp lệ nếu `r + 1 ≤ L`;
- `d ≠ c` ⇒ `(d, 1)`.

Có `2L` trạng thái, mỗi trạng thái 2 chuyển ⇒ `O(n · L)` cho một `L`.

### 3.3. Tính đơn điệu

Nếu đạt được "mọi khối ≤ `L`" với `t` lần lật thì cũng đạt "≤ `L + 1`" với `t` lần ⇒ `minFlips(L)` không tăng theo `L` ⇒ tìm `L` nhỏ nhất có `minFlips(L) ≤ numOps` bằng nhị phân.

In [3]:
def min_flips(s, L):
    INF = float('inf')
    cur = [[INF] * (L + 1) for _ in range(2)]        # cur[c][r]
    b0 = int(s[0])
    cur[b0][1] = 0
    cur[1 - b0][1] = 1
    for ch in s[1:]:
        b = int(ch)
        nxt = [[INF] * (L + 1) for _ in range(2)]
        for c in range(2):
            for r in range(1, L + 1):
                v = cur[c][r]
                if v == INF:
                    continue
                if r + 1 <= L:                          # giữ ký tự c
                    w = v + (b != c)
                    if w < nxt[c][r + 1]:
                        nxt[c][r + 1] = w
                w = v + (b != 1 - c)                    # đổi sang 1 - c
                if w < nxt[1 - c][1]:
                    nxt[1 - c][1] = w
        cur = nxt
    return min(min(row) for row in cur)

print([min_flips("000001", L) for L in range(1, 7)])

[2, 1, 1, 1, 0, 0]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Ví dụ 1 `"000001"`: `minFlips(L)` cho `L = 1..6` là `[3, 1, 1, 0, 0, 0]` (ô trên). Với `numOps = 1`, `L` nhỏ nhất có `minFlips ≤ 1` là `2` ✓.

Ví dụ 2 `"0000"`, `numOps = 2`: `minFlips(1) = 2` (thành `"1010"` hoặc `"0101"`) ⇒ đáp án 1.

In [4]:
print('"0000":', [min_flips("0000", L) for L in range(1, 5)])

"0000": [2, 1, 1, 0]


## 5. Cách 1 — Brute Force: thử mọi tập vị trí lật

In [5]:
import itertools

def min_len_brute(s, k):
    n, best = len(s), len(s)
    for r in range(min(k, n) + 1):
        for pos in itertools.combinations(range(n), r):
            t = list(s)
            for p in pos:
                t[p] = '1' if t[p] == '0' else '0'
            best = min(best, longest_run(t))
    return best

for s, k, exp in examples:
    print(min_len_brute(s, k), '| kỳ vọng', exp)

2 | kỳ vọng 2
1 | kỳ vọng 1
1 | kỳ vọng 1


**Độ phức tạp:** mũ — chỉ cho `n ≤ 12`.

## 6. Ý tưởng thuật toán / bước đột phá

Nhị phân `L ∈ [1, n]`, kiểm tra `min_flips(s, L) ≤ numOps` bằng DP mục 3.2. Tổng `O(n · L · log n) ≤ O(n² log n) ≈ 10^7` với `n = 1000` — chấp nhận được (thực tế `L` nhỏ dần trong nhị phân).

## 7. Các cách tiếp theo

### 7.1. Cách 2 — DP cho mọi `L` tăng dần, không nhị phân

In [6]:
def min_len_linear(s, k):
    for L in range(1, len(s) + 1):
        if min_flips(s, L) <= k:
            return L
    return len(s)

for s, k, exp in examples:
    print(min_len_linear(s, k), '| kỳ vọng', exp)

2 | kỳ vọng 2
1 | kỳ vọng 1
1 | kỳ vọng 1


`O(n · Σ L) = O(n · ans²)` — tốt khi đáp án nhỏ, tệ khi đáp án lớn.

### 7.2. Cách 3 — Nhị phân + DP (lời giải chính cho bản I)

In [7]:
def min_len_bs_dp(s, k):
    lo, hi = 1, len(s)
    while lo < hi:
        mid = (lo + hi) // 2
        if min_flips(s, mid) <= k:
            hi = mid
        else:
            lo = mid + 1
    return lo

for s, k, exp in examples:
    print(min_len_bs_dp(s, k), '| kỳ vọng', exp)

2 | kỳ vọng 2
1 | kỳ vọng 1
1 | kỳ vọng 1


### 7.3. Cách 4 — Công thức theo run (dùng cho bản II, LC 3399)

`L = 1`: chi phí bằng khoảng cách tới `0101…`/`1010…`. `L ≥ 2`: `Σ_run ⌊len / (L+1)⌋`. Ở đây ta dùng DP để **kiểm chứng** công thức trên rất nhiều xâu.

In [8]:
def runs(s):
    out, i = [], 0
    while i < len(s):
        j = i
        while j < len(s) and s[j] == s[i]:
            j += 1
        out.append(j - i); i = j
    return out

def formula_flips(s, L):
    if L == 1:
        a = sum(1 for i, c in enumerate(s) if c != '01'[i % 2])
        return min(a, len(s) - a)
    return sum(r // (L + 1) for r in runs(s))

def min_len_formula(s, k):
    lo, hi = 1, len(s)
    while lo < hi:
        mid = (lo + hi) // 2
        if formula_flips(s, mid) <= k: hi = mid
        else: lo = mid + 1
    return lo

for s, k, exp in examples:
    print(min_len_formula(s, k), '| kỳ vọng', exp)

2 | kỳ vọng 2
1 | kỳ vọng 1
1 | kỳ vọng 1


## 8. TRACE từng bước

Trace DP với `s = "00000"`, `L = 2`: in bảng chi phí ít nhất cho mỗi trạng thái `(c, r)` sau mỗi ký tự.

In [9]:
def trace_dp(s, L):
    INF = float('inf')
    cur = {(c, 1): (0 if s[0] == c else 1) for c in '01'}
    fmt = lambda st: ', '.join(f"{c}{r}:{v}" for (c, r), v in sorted(st.items()))
    print(f"sau s[0]={s[0]!r}: {fmt(cur)}")
    for i, ch in enumerate(s[1:], 1):
        nxt = {}
        for (c, r), v in cur.items():
            for d in '01':
                if d == c and r + 1 > L:
                    continue
                key = (d, r + 1) if d == c else (d, 1)
                nxt[key] = min(nxt.get(key, INF), v + (ch != d))
        cur = nxt
        print(f"sau s[{i}]={ch!r}: {fmt(cur)}")
    print('min lật =', min(cur.values()))

trace_dp("00000", 2)

sau s[0]='0': 01:0, 11:1
sau s[1]='0': 01:1, 02:0, 11:1, 12:2
sau s[2]='0': 01:1, 02:1, 11:1, 12:2
sau s[3]='0': 01:1, 02:1, 11:2, 12:2
sau s[4]='0': 01:2, 02:1, 11:2, 12:3
min lật = 1


Đọc trạng thái `"0 2: 1"` là: kết thúc bằng hai số `0` liên tiếp, đã lật 1 lần.

## 9. Code nộp LeetCode

In [10]:
class Solution:
    def minLength(self, s: str, numOps: int) -> int:
        n = len(s)
        bits = [ord(c) - 48 for c in s]
        INF = n + 1

        def flips(L):
            cur = [[INF] * (L + 1) for _ in range(2)]
            cur[bits[0]][1] = 0
            cur[1 - bits[0]][1] = 1
            for b in bits[1:]:
                nxt = [[INF] * (L + 1) for _ in range(2)]
                for c in range(2):
                    row = cur[c]
                    best_any = INF
                    for r in range(1, L + 1):
                        v = row[r]
                        if v < best_any:
                            best_any = v
                        if r < L and v < INF:
                            w = v + (b != c)
                            if w < nxt[c][r + 1]:
                                nxt[c][r + 1] = w
                    if best_any < INF:
                        w = best_any + (b == c)          # đổi sang ký tự 1 - c
                        if w < nxt[1 - c][1]:
                            nxt[1 - c][1] = w
                cur = nxt
            return min(min(cur[0]), min(cur[1]))

        lo, hi = 1, n
        while lo < hi:
            mid = (lo + hi) // 2
            if flips(mid) <= numOps:
                hi = mid
            else:
                lo = mid + 1
        return lo

for s, k, exp in examples:
    got = Solution().minLength(s, k)
    print(got, 'OK' if got == exp else 'SAI')

2 OK
1 OK
1 OK


Bản nộp gộp chuyển "đổi ký tự" bằng **min theo hàng** (`best_any`) — vì từ mọi `r` đều sang cùng trạng thái `(1 − c, 1)`.

## 10. Vì sao thuật toán đúng

1. **DP đúng:** mọi xâu kết quả `t` (sau lật) với khối ≤ `L` tương ứng một đường đi trong máy trạng thái (trạng thái sau ký tự `i` là `(t_i, độ dài khối cuối)`), chi phí = số vị trí `t_i ≠ s_i`. Ngược lại mọi đường đi hợp lệ cho một xâu `t` có khối ≤ `L`. ⇒ min chi phí đường đi = số lần lật ít nhất.
2. **Đơn điệu (3.3)** ⇒ nhị phân trả về `L` nhỏ nhất.
3. **Mỗi lần lật chỉ dùng một lần trên mỗi vị trí:** lật hai lần cùng vị trí là vô ích, nên "số lần lật" = số vị trí khác nhau.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Thử mọi tập lật | mũ | `O(n)` |
| 2. DP với `L` tăng dần | `O(n · ans²)` | `O(ans)` |
| 3. Nhị phân + DP | `O(n · L · log n)` ≤ `O(n² log n)` | `O(L)` |
| 4. Công thức run | `O(n log n)` | `O(n)` |

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [11]:
impls = {'linear': min_len_linear, 'bs_dp': min_len_bs_dp, 'formula': min_len_formula,
         'submit': lambda s, k: Solution().minLength(s, k)}
edge = [("0", 0, 1), ("01", 0, 1), ("00", 0, 2), ("0110", 1, 2), ("0110", 2, 1), ("1111", 1, 2), ("000111000", 2, 3)]   # ba run dài 3, L = 2 cần 3 lần lật
for s, k, exp in examples + edge:
    assert min_len_brute(s, k) == exp, (s, k, min_len_brute(s, k))
    for name, f in impls.items():
        assert f(s, k) == exp, (name, s, k)
print('Ví dụ đề và ca biên: tất cả đúng.')

Ví dụ đề và ca biên: tất cả đúng.


### 12.2. Đối chiếu ngẫu nhiên; kiểm chứng công thức bằng DP

In [12]:
import random
rng = random.Random(3398)
cnt = 0
for _ in range(1500):
    n = rng.randint(1, 11)
    s = ''.join(rng.choice('01') if rng.random() < 0.5 else '1' for _ in range(n))
    k = rng.randint(0, n)
    exp = min_len_brute(s, k)
    for name, f in impls.items():
        assert f(s, k) == exp, (name, s, k)
    cnt += 1
mismatch = 0
for _ in range(500):
    s = ''.join(rng.choice('0001') for _ in range(rng.randint(1, 60)))
    for L in range(1, 8):
        mismatch += formula_flips(s, L) != min_flips(s, L)
print('Khớp vét cạn trên', cnt, 'bộ; công thức run khớp DP trên 3500 cặp (s, L), số lệch =', mismatch)

Khớp vét cạn trên 1500 bộ; công thức run khớp DP trên 3500 cặp (s, L), số lệch = 0


### 12.3. Ca lớn `n = 1000`

In [13]:
import time
n = 1000
cases = {
    'toàn 0, k = 0': ('0' * n, 0),
    'toàn 0, k = 3': ('0' * n, 3),
    'ngẫu nhiên lệch, k = 50': (''.join(rng.choice('0001') for _ in range(n)), 50),
    'xen kẽ, k = 0': (''.join('01'[i % 2] for i in range(n)), 0),
}
for name, (s, k) in cases.items():
    t = time.time(); a = Solution().minLength(s, k); t1 = time.time() - t
    t = time.time(); b = min_len_formula(s, k); t2 = time.time() - t
    assert a == b
    print(f"{name:26s}: {a:5d} | nhị phân + DP {t1:.2f}s | công thức {t2:.4f}s")

toàn 0, k = 0             :  1000 | nhị phân + DP 1.15s | công thức 0.0004s


toàn 0, k = 3             :   250 | nhị phân + DP 0.37s | công thức 0.0004s
ngẫu nhiên lệch, k = 50   :     5 | nhị phân + DP 0.17s | công thức 0.0009s


xen kẽ, k = 0             :     1 | nhị phân + DP 0.16s | công thức 0.0013s


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — DP cho phép khối dài `L + 1` (điều kiện `r ≤ L` thay vì `r + 1 ≤ L`)

In [14]:
def wrong_flips(s, L):
    INF = float('inf')
    cur = {(c, 1): (0 if s[0] == c else 1) for c in '01'}
    for ch in s[1:]:
        nxt = {}
        for (c, r), v in cur.items():
            for d in '01':
                if d == c and r > L:                     # SAI: cho phép r + 1 = L + 1
                    continue
                key = (d, r + 1) if d == c else (d, 1)
                nxt[key] = min(nxt.get(key, INF), v + (ch != d))
        cur = nxt
    return min(cur.values())

def wrong_len(s, k):
    return next(L for L in range(1, len(s) + 1) if wrong_flips(s, L) <= k)

print('sai :', wrong_len("000001", 1), '| đúng:', Solution().minLength("000001", 1))

sai : 1 | đúng: 2


**Vì sao sai:** trạng thái `r` đã là độ dài khối; thêm một ký tự giống nhau tạo khối `r + 1`, phải kiểm `r + 1 ≤ L`.

### Lỗi 2 — Khởi tạo chỉ trạng thái "giữ nguyên" `s[0]` (quên khả năng lật ký tự đầu)

In [15]:
def wrong_init_flips(s, L):
    INF = float('inf')
    cur = {(s[0], 1): 0}                                  # SAI: thiếu (ký tự lật, 1): 1
    for ch in s[1:]:
        nxt = {}
        for (c, r), v in cur.items():
            for d in '01':
                if d == c and r + 1 > L:
                    continue
                key = (d, r + 1) if d == c else (d, 1)
                nxt[key] = min(nxt.get(key, INF), v + (ch != d))
        cur = nxt
    return min(cur.values())

print('min lật "001" với L=1: sai =', wrong_init_flips("001", 1), '| đúng =', min_flips("001", 1))

min lật "001" với L=1: sai = 2 | đúng = 1


**Vì sao sai:** đôi khi lời giải tốt nhất phải lật **chính ký tự đầu**: `"001" → "101"` chỉ tốn 1 lần, còn nếu giữ nguyên ký tự đầu thì phải thành `"010"` (2 lần). Bỏ trạng thái "lật ký tự đầu" làm chi phí bị đánh giá cao ⇒ đáp án có thể lớn hơn thực tế.

### Lỗi 3 — Nhị phân với cận dưới `0` hoặc trả `lo − 1`

In [16]:
def wrong_bounds(s, k):
    lo, hi = 0, len(s)
    while lo < hi:
        mid = (lo + hi + 1) // 2
        if min_flips(s, mid) > k:                       # SAI: tìm L lớn nhất KHÔNG đạt rồi quên +1
            lo = mid
        else:
            hi = mid - 1
    return lo

print('sai :', wrong_bounds("000001", 1), '| đúng:', Solution().minLength("000001", 1))

sai : 1 | đúng: 2


**Vì sao sai:** nhị phân này tìm `L` lớn nhất **không** đạt; đáp án là giá trị đó **cộng 1**. Viết rõ "tìm `L` nhỏ nhất đạt" như bản nộp để tránh nhầm.

### Lỗi 4 — Coi `numOps` là số lần lật **bắt buộc** (phải dùng hết)

In [17]:
def wrong_exact_ops(s, k):
    n, best = len(s), len(s)
    for pos in itertools.combinations(range(n), min(k, n)):
        t = list(s)
        for p in pos:
            t[p] = '1' if t[p] == '0' else '0'
        best = min(best, longest_run(t))
    return best

print('sai :', wrong_exact_ops("0101", 2), '| đúng:', Solution().minLength("0101", 2))

sai : 2 | đúng: 1


**Vì sao sai:** đề cho phép **tối đa** `numOps` lần. Bắt buộc lật đúng `k` vị trí khác nhau có thể phá hỏng xâu đang tốt (`"0101"`).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Tối thiểu hóa max" ⇒ nhị phân + hàm kiểm tra.
- Ràng buộc "khối ≤ `L`" ⇒ DP máy trạng thái `(ký tự cuối, độ dài khối)`.
- Gộp các chuyển giống nhau (đổi ký tự từ mọi `r`) bằng min theo hàng.
- `n ≤ 1000` cho phép `O(n² log n)`; khi `n` lớn (bản II) cần công thức.

## 15. Biến thể của bài

1. **LC 3399 — bản II** (`n ≤ 10^5`): công thức run + trường hợp `L = 1`.
2. Lật có **chi phí khác nhau** theo vị trí ⇒ DP vẫn đúng (thay chi phí 1 bằng `w_i`), công thức run thì không.
3. Bảng chữ cái `σ` ký tự ⇒ trạng thái `(c, r)` với `σ` giá trị.
4. Thêm ràng buộc khối ≥ `K` ⇒ DP tương tự LC 3441.

In [18]:
# Biến thể 2: lật vị trí i tốn w[i]; tìm L nhỏ nhất với tổng chi phí <= budget
def min_len_weighted(s, w, budget):
    def cost(L):
        INF = float('inf')
        cur = {(c, 1): (0 if s[0] == c else w[0]) for c in '01'}
        for i in range(1, len(s)):
            nxt = {}
            for (c, r), v in cur.items():
                for d in '01':
                    if d == c and r + 1 > L:
                        continue
                    key = (d, r + 1) if d == c else (d, 1)
                    nxt[key] = min(nxt.get(key, INF), v + (w[i] if s[i] != d else 0))
            cur = nxt
        return min(cur.values())
    return next(L for L in range(1, len(s) + 1) if cost(L) <= budget)

def brute_weighted(s, w, budget):
    n, best = len(s), len(s)
    for mask in range(1 << n):
        c = sum(w[i] for i in range(n) if mask >> i & 1)
        if c <= budget:
            t = [('1' if s[i] == '0' else '0') if mask >> i & 1 else s[i] for i in range(n)]
            best = min(best, longest_run(t))
    return best

rng = random.Random(15)
for _ in range(300):
    n = rng.randint(1, 9)
    s = ''.join(rng.choice('001') for _ in range(n)); w = [rng.randint(1, 4) for _ in range(n)]
    budget = rng.randint(0, 8)
    assert min_len_weighted(s, w, budget) == brute_weighted(s, w, budget)
print('Biến thể chi phí theo vị trí khớp vét cạn.')

Biến thể chi phí theo vị trí khớp vét cạn.


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Ghi chú |
|---|---|---|---|
| 1 | Thử mọi tập lật | mũ | Oracle |
| 2 | DP, `L` tăng dần | `O(n · ans²)` | Tốt khi đáp án nhỏ |
| 3 | Nhị phân + DP | `O(n² log n)` | **Nộp bài (bản I)** |
| 4 | Công thức run | `O(n log n)` | Dùng cho bản II |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `"1100"`, `numOps = 1`.

**Bài 2 (tính tay):** bảng `min_flips("0000000", L)` với `L = 1..4`.

**Bài 3 (điền TODO):** viết `reconstruct(s, L)` — xâu kết quả đạt `min_flips(s, L)` (truy vết DP).

**Bài 4 (tự viết lại):** DP cho bảng chữ cái 3 ký tự `'a','b','c'`.

**Bài 5 (mini challenge):** đếm số xâu kết quả **khác nhau** đạt số lần lật tối thiểu với `L` cho trước.

In [19]:
def reconstruct_todo(s, L):
    # TODO
    raise NotImplementedError

def min_flips_ternary_todo(s, L):
    # TODO
    raise NotImplementedError

def count_optimal_results_todo(s, L):
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) `L = 1` cần xen kẽ: `"1010"` hoặc `"0101"` — khác `"1100"` ở 2 vị trí. (2) `L = 2` đã thỏa. (3) Đáp án 2.

**Bài 2.** (1) `L = 1`: xen kẽ, 3 lần. (2) `L = 2`: `⌊7/3⌋ = 2`. (3) `L = 3`: 1; `L = 4`: 1.

**Bài 3.** (1) Lưu trạng thái cha cho mỗi `(i, c, r)`. (2) Từ trạng thái cuối tốt nhất đi ngược. (3) Kiểm tra số vị trí khác và khối dài nhất.

**Bài 4.** (1) `c ∈ {a, b, c}`. (2) Đổi sang ký tự khác: 2 lựa chọn. (3) Đối chiếu vét cạn.

**Bài 5.** (1) DP cặp `(chi phí, số cách)`. (2) Gộp bằng min, cộng số cách khi hòa. (3) Vét cạn kiểm tra.

## 19. Đáp án tham khảo cho bài tập

In [20]:
# Bài 1, 2
assert Solution().minLength("1100", 1) == 2 == min_len_brute("1100", 1)
table = [min_flips("0000000", L) for L in range(1, 5)]
assert table == [3, 2, 1, 1]
print('Bài 1: 2 | Bài 2:', table)

# Bài 3
def reconstruct(s, L):
    INF = float('inf')
    n = len(s)
    layers = [{(c, 1): ((0 if s[0] == c else 1), None) for c in '01'}]
    for i in range(1, n):
        nxt = {}
        for (c, r), (v, _) in layers[-1].items():
            for d in '01':
                if d == c and r + 1 > L:
                    continue
                key = (d, r + 1) if d == c else (d, 1)
                w = v + (s[i] != d)
                if w < nxt.get(key, (INF, None))[0]:
                    nxt[key] = (w, (c, r))
        layers.append(nxt)
    state = min(layers[-1], key=lambda st: layers[-1][st][0])
    out = []
    for i in range(n - 1, -1, -1):
        out.append(state[0])
        state = layers[i][state][1]
    return ''.join(reversed(out))

rng = random.Random(3)
for _ in range(300):
    s = ''.join(rng.choice('001') for _ in range(rng.randint(1, 30))); L = rng.randint(1, 5)
    t = reconstruct(s, L)
    assert longest_run(t) <= L and sum(a != b for a, b in zip(s, t)) == min_flips(s, L)
print('Bài 3: truy vết hợp lệ; "000001", L=2 ->', reconstruct("000001", 2))

Bài 1: 2 | Bài 2: [3, 2, 1, 1]
Bài 3: truy vết hợp lệ; "000001", L=2 -> 001001


In [21]:
# Bài 4, 5
def min_flips_ternary(s, L):
    INF = float('inf')
    cur = {(c, 1): (0 if s[0] == c else 1) for c in 'abc'}
    for ch in s[1:]:
        nxt = {}
        for (c, r), v in cur.items():
            for d in 'abc':
                if d == c and r + 1 > L:
                    continue
                key = (d, r + 1) if d == c else (d, 1)
                nxt[key] = min(nxt.get(key, INF), v + (ch != d))
        cur = nxt
    return min(cur.values())

def count_optimal_results(s, L):
    cur = {}
    for c in '01':
        cur[(c, 1)] = ((0 if s[0] == c else 1), 1)
    for ch in s[1:]:
        nxt = {}
        for (c, r), (v, ways) in cur.items():
            for d in '01':
                if d == c and r + 1 > L:
                    continue
                key = (d, r + 1) if d == c else (d, 1)
                w = v + (ch != d)
                if key not in nxt or w < nxt[key][0]:
                    nxt[key] = (w, ways)
                elif w == nxt[key][0]:
                    nxt[key] = (w, nxt[key][1] + ways)
        cur = nxt
    best = min(v for v, _ in cur.values())
    return sum(ways for v, ways in cur.values() if v == best)

def brute_count(s, L):
    n = len(s); res = {}
    for t in itertools.product('01', repeat=n):
        if longest_run(t) <= L:
            res.setdefault(sum(a != b for a, b in zip(s, t)), []).append(t)
    return len(res[min(res)])

rng = random.Random(5)
for _ in range(200):
    s = ''.join(rng.choice('01') for _ in range(rng.randint(1, 9))); L = rng.randint(1, 4)
    assert count_optimal_results(s, L) == brute_count(s, L)
    s3 = ''.join(rng.choice('aab') for _ in range(rng.randint(1, 6)))
    best = min(sum(a != b for a, b in zip(s3, t)) for t in itertools.product('abc', repeat=len(s3)) if longest_run(t) <= L)
    assert min_flips_ternary(s3, L) == best
print('Bài 4, 5 khớp vét cạn; "0000", L=1 có', count_optimal_results("0000", 1), 'xâu tối ưu.')

Bài 4, 5 khớp vét cạn; "0000", L=1 có 2 xâu tối ưu.


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
def flips(L):
    cur[c][r] (r = 1..L);  khởi tạo cả (s0, 1): 0 và (1 - s0, 1): 1
    for b in s[1:]:
        giữ c:  r + 1 <= L, cost += (b != c)
        đổi:    (1 - c, 1) từ min_r cur[c][r], cost += (b == c)
    return min
nhị phân L nhỏ nhất với flips(L) <= numOps
```

**Checklist:**

- [ ] Kiểm `r + 1 ≤ L` khi nối thêm cùng ký tự?
- [ ] Khởi tạo cả trạng thái lật ký tự đầu?
- [ ] Nhị phân tìm `L` **nhỏ nhất** đạt?
- [ ] "Tối đa" `numOps` (không bắt buộc dùng hết)?
- [ ] `O(n² log n)` chỉ dùng được cho bản `n ≤ 1000`?